# Proyecto ML - Parte 1: Machine Learning Clásico

Este notebook construye la entrega final de la Parte 1 del proyecto. El objetivo es clasificar textos según su década de origen usando únicamente técnicas clásicas de aprendizaje automático con `scikit-learn`.

De acuerdo con el enunciado, no se usan redes neuronales, aprendizaje profundo, transformers, embeddings preentrenados ni modelos de lenguaje. Siguiendo el hint recibido para esta parte, el modelo final es una regresión logística dentro de un `Pipeline` con representación TF-IDF.


In [1]:
import os

# Limita hilos para que el notebook sea estable en entornos locales o de evaluación.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Carga de datos

El conjunto `train.csv` contiene los textos etiquetados con `decade`. El conjunto `eval.csv` contiene los textos sin etiqueta y se usa únicamente para generar el archivo de respuesta de Kaggle.


In [2]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / "Data" / "train.csv").exists():
    BASE_DIR = BASE_DIR / "Proyecto-ML"

TRAIN_PATH = BASE_DIR / "Data" / "train.csv"
EVAL_PATH = BASE_DIR / "Data" / "eval.csv"
SUBMISSION_PATH = BASE_DIR / "submission_parte1.csv"
MODEL_PATH = BASE_DIR / "modelo_parte1.joblib"

train_df = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)

print(f"train.csv: {train_df.shape}")
print(f"eval.csv: {eval_df.shape}")
print("Columnas train:", train_df.columns.tolist())
print("Columnas eval:", eval_df.columns.tolist())

display(train_df.head())
display(eval_df.head())


train.csv: (31403, 2)
eval.csv: (3490, 2)
Columnas train: ['text', 'decade']
Columnas eval: ['id', 'text']


,text,decade
0,\nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...,164
1,"gone. Sus amigos , sus clientes, todo \ncuanto...",182
2,"Prefosen quemanera,e per qualesfolpechas deuan...",157
3,Caistro el M a y o r a i .] Del ape...,163
4,\nlos que panden macho ; y \notros en l...,166


,id,text
0,0,P. Si en efta convocación trato folament...
1,1,«Muy santo Padre : Ayer escribiá don Juan Man-...
2,2,"Recibo, otorgado por Diego Gracián a favor de ..."
3,3,6. Los Samaritanos no admitían por \nEfcr...
4,4,ü«mi yofacaremisoiiejasfilasmaí \nI nosodlosr...


## 2. Exploración y limpieza mínima de datos

Antes de entrenar el modelo se revisan las columnas esperadas, tamaños, textos nulos o vacíos, etiquetas faltantes, duplicados y distribución de clases. La limpieza se limita a eliminar registros de entrenamiento sin texto o sin etiqueta. No se modifican acentos, grafías antiguas, signos, mayúsculas/minúsculas manualmente ni caracteres raros, porque esas variaciones pueden ser señales útiles para diferenciar décadas.


In [3]:
expected_train_cols = {"text", "decade"}
expected_eval_cols = {"id", "text"}

assert expected_train_cols.issubset(train_df.columns), "train.csv no tiene las columnas esperadas: text, decade"
assert expected_eval_cols.issubset(eval_df.columns), "eval.csv no tiene las columnas esperadas: id, text"

quality_summary = pd.DataFrame({
    "conjunto": ["train", "eval"],
    "filas": [len(train_df), len(eval_df)],
    "textos_nulos": [train_df["text"].isna().sum(), eval_df["text"].isna().sum()],
    "textos_vacios": [train_df["text"].fillna("").str.strip().eq("").sum(), eval_df["text"].fillna("").str.strip().eq("").sum()],
    "etiquetas_o_ids_nulos": [train_df["decade"].isna().sum(), eval_df["id"].isna().sum()],
    "duplicados_exactos": [train_df.duplicated().sum(), eval_df.duplicated().sum()],
})

display(quality_summary)

# Limpieza mínima del conjunto de entrenamiento: solo se eliminan filas sin texto o sin etiqueta.
train_model_df = train_df.dropna(subset=["text", "decade"]).copy()
train_model_df = train_model_df[train_model_df["text"].str.strip().ne("")].copy()
train_model_df["decade"] = train_model_df["decade"].astype(int)

# En evaluación no se eliminan filas porque Kaggle exige una predicción para cada id.
eval_model_df = eval_df.copy()
eval_model_df["text"] = eval_model_df["text"].fillna("")

print(f"Filas de train antes de limpieza: {len(train_df)}")
print(f"Filas de train después de limpieza mínima: {len(train_model_df)}")
print(f"Filas de eval conservadas: {len(eval_model_df)}")
print("Número de clases:", train_model_df["decade"].nunique())
print("Rango de décadas:", train_model_df["decade"].min(), "a", train_model_df["decade"].max())

display(train_model_df["decade"].value_counts().sort_index().to_frame("conteo").T)

text_lengths = train_model_df["text"].str.len()
length_summary = text_lengths.describe()[["min", "25%", "50%", "75%", "max", "mean"]].to_frame("longitud_texto").T
display(length_summary)


,conjunto,filas,textos_nulos,textos_vacios,etiquetas_o_ids_nulos,duplicados_exactos
0,train,31403,0,0,0,34
1,eval,3490,0,0,0,0


Filas de train antes de limpieza: 31403
Filas de train después de limpieza mínima: 31403
Filas de eval conservadas: 3490
Número de clases: 39
Rango de décadas: 150 a 188


decade,150,151,152,153,154,155,156,157,158,159,...,179,180,181,182,183,184,185,186,187,188
conteo,786,812,785,775,830,836,792,827,778,802,...,809,825,795,808,794,802,803,773,787,809


,min,25%,50%,75%,max,mean
longitud_texto,120.0,182.0,315.0,643.0,7418.0,520.56829


## 3. Preparación de datos

La partición local se usa solo para estimar el desempeño del pipeline final antes de entrenarlo con todos los datos etiquetados limpios. `eval.csv` no se usa en entrenamiento ni validación; únicamente se conserva para generar el archivo final con todos los `id`.


In [4]:
X = train_model_df["text"]
y = train_model_df["decade"]
X_eval = eval_model_df["text"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Tamaño entrenamiento local:", X_train.shape[0])
print("Tamaño validación local:", X_val.shape[0])


Tamaño entrenamiento local: 25122
Tamaño validación local: 6281


## 4. Pipeline final

El pipeline final usa `TfidfVectorizer` con n-gramas de caracteres y `LogisticRegression`. Esta representación es adecuada para textos históricos porque captura patrones ortográficos y es robusta ante variaciones de escritura o ruido en el texto.


In [5]:
final_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        sublinear_tf=True,
        min_df=2,
        max_df=0.95,
        max_features=200_000,
        lowercase=True,
    )),
    ("model", LogisticRegression(
        C=4.0,
        penalty="l2",
        solver="liblinear",
        dual=True,
        max_iter=2000,
        random_state=RANDOM_STATE,
    )),
])

final_pipeline


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char', max_df=0.95,
                                 max_features=200000, min_df=2,
                                 ngram_range=(3, 5), sublinear_tf=True)),
                ('model',
                 LogisticRegression(C=4.0, dual=True, max_iter=2000,
                                    random_state=42, solver='liblinear'))])

## 5. Entrenamiento y evaluación local

Se entrena el pipeline final en la partición local de entrenamiento y se evalúa sobre la partición de validación.


In [6]:
final_pipeline.fit(X_train, y_train)

y_pred_train = final_pipeline.predict(X_train)
y_pred_val = final_pipeline.predict(X_val)

metrics = pd.DataFrame({
    "conjunto": ["train", "validacion"],
    "accuracy": [
        accuracy_score(y_train, y_pred_train),
        accuracy_score(y_val, y_pred_val),
    ],
    "f1_macro": [
        f1_score(y_train, y_pred_train, average="macro"),
        f1_score(y_val, y_pred_val, average="macro"),
    ],
})

display(metrics)
print(classification_report(y_val, y_pred_val, zero_division=0))


,conjunto,accuracy,f1_macro
0,train,0.992118,0.992103
1,validacion,0.280051,0.274033


              precision    recall  f1-score   support

         150       0.76      0.76      0.76       157
         151       0.48      0.73      0.58       162
         152       0.66      0.60      0.63       157
         153       0.60      0.70      0.65       155
         154       0.51      0.58      0.54       166
         155       0.37      0.31      0.34       167
         156       0.38      0.42      0.40       158
         157       0.29      0.34      0.31       166
         158       0.28      0.31      0.29       156
         159       0.23      0.29      0.26       160
         160       0.15      0.12      0.13       170
         161       0.12      0.11      0.12       157
         162       0.22      0.20      0.21       162
         163       0.19      0.18      0.18       166
         164       0.24      0.20      0.22       161
         165       0.10      0.09      0.09       163
         166       0.11      0.07      0.09       156
         167       0.20    

El desempeño en validación se usa como estimación local. Para el envío final, el mismo pipeline se reentrena con todo `train.csv`, como corresponde para aprovechar todos los ejemplos etiquetados disponibles.


## 6. Entrenamiento final y archivo para Kaggle

Se reentrena el pipeline final con todos los datos etiquetados y se generan predicciones para `eval.csv`. El archivo de respuesta debe tener exactamente las columnas `id,answer`.


In [7]:
final_pipeline.fit(X, y)

eval_predictions = final_pipeline.predict(X_eval)
submission_df = pd.DataFrame({
    "id": eval_model_df["id"],
    "answer": eval_predictions.astype(int),
})

assert list(submission_df.columns) == ["id", "answer"]
assert len(submission_df) == len(eval_df)
assert submission_df.isna().sum().sum() == 0
assert set(submission_df["answer"]).issubset(set(y.unique()))

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Archivo guardado en: {SUBMISSION_PATH}")
print(f"Filas: {len(submission_df)}")
display(submission_df.head())


Archivo guardado en: /Users/pablogalindo/Desktop/UNIVERSIDAD/NOVENO SEMESTRE/MACHINE LEARNING/Laboratorios/Proyecto-ML/submission_parte1.csv
Filas: 3490


,id,answer
0,0,173
1,1,185
2,2,150
3,3,169
4,4,153


## 7. Guardado del modelo final

El enunciado pide entregar un único modelo final de `scikit-learn`, correspondiente al mejor envío de Kaggle. Se guarda el pipeline completo para que incluya tanto la vectorización TF-IDF como el clasificador.


In [8]:
joblib.dump(final_pipeline, MODEL_PATH)
loaded_pipeline = joblib.load(MODEL_PATH)

sample_predictions = loaded_pipeline.predict(X_eval.head(5))
print(f"Modelo guardado en: {MODEL_PATH}")
print("Predicciones de verificación:", sample_predictions)


Modelo guardado en: /Users/pablogalindo/Desktop/UNIVERSIDAD/NOVENO SEMESTRE/MACHINE LEARNING/Laboratorios/Proyecto-ML/modelo_parte1.joblib
Predicciones de verificación: [173 185 150 169 153]


## 8. Conclusión

El modelo final de la Parte 1 es un `Pipeline` de `scikit-learn` compuesto por `TfidfVectorizer` con n-gramas de caracteres y `LogisticRegression`. Este notebook genera los dos archivos necesarios para la entrega final:

- `submission_parte1.csv`: archivo para subir a Kaggle con columnas `id,answer`.
- `modelo_parte1.joblib`: pipeline final entrenado, para entregar en Bloque Neón junto con este notebook.
